# Leave-One-Out Baseline

Non-gradient baseline for pairwise ranking explanations. For each document word, mask that word, recompute the pairwise decision score, and use the score change as a support value. This is a lightweight leave-one-out baseline inspired by a SHAP-like local effect test.


In [1]:
# -- IMPORTS --
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sentence_transformers import CrossEncoder


/Users/evelinalune/Documents/uni/MSc-IS/thesis/ig-thesis/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# experiment_name = "cross_encoder"
# experiment_name = "monot5"
experiment_name = "duot5"

EXPERIMENTS = {
    "cross_encoder": {
        "label": "Cross-encoder",
        "family": "cross_encoder",
        "model_name": "cross-encoder/ms-marco-MiniLM-L-6-v2",
        "out_dir": Path("../outputs"),
        "score_label": "g(q, d_i, d_j)",
        "mask_strategy": "mask",
    },
    "monot5": {
        "label": "monoT5",
        "family": "monot5",
        "model_name": "castorini/monot5-base-msmarco",
        "out_dir": Path("../outputs_monot5"),
        "score_label": "pointwise true-probability difference",
        "mask_strategy": "pad",
    },
    "duot5": {
        "label": "DuoT5",
        "family": "duot5",
        "model_name": "castorini/duot5-base-msmarco",
        "out_dir": Path("../outputs_t5"),
        "score_label": "pairwise margin (true - false)",
        "mask_strategy": "pad",
    },
}

cfg = EXPERIMENTS[experiment_name]
model_family = cfg["family"]
model_name = cfg["model_name"]
out_dir = cfg["out_dir"]
pair_paths = [
    out_dir / "pairwise_scores.parquet",
    out_dir / "pairwise_scores.pkl",
    out_dir / "pairwise_scores.csv",
]
loo_out = out_dir / "loo_attributions.pkl"
loo_summary_out = out_dir / "loo_attributions_summary.csv"

max_length = 512
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
print(f"Experiment: {cfg['label']} ({experiment_name})")


Experiment: DuoT5 (duot5)


In [3]:
def load_pairs(paths):
    for path in paths:
        if not path.exists():
            continue
        if path.suffix == ".parquet":
            try:
                return pd.read_parquet(path)
            except ImportError:
                continue
        if path.suffix in {".pkl", ".pickle"}:
            return pd.read_pickle(path)
        if path.suffix == ".csv":
            return pd.read_csv(path)
    raise FileNotFoundError("No readable pairwise score file was found.")

pairs_df = load_pairs(pair_paths)
print(f"Loaded {len(pairs_df):,} pairwise rows")
pairs_df.head()


Loaded 80 pairwise rows


,qid,query,pid_i,passage_i,score_i,pid_j,passage_j,score_j,g_score,correct_pref
0,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,4083953,Cost to Attend. The total cost to attend inclu...,0.384977,0.615020,1
1,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,2281863,"If you're paying for college, you will save li...",0.389508,0.610489,1
2,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,6262988,Undergraduate Tuition. Southern Illinois Unive...,0.232101,0.767897,1
3,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,6650510,"The overall cost for on-campus, in-state stude...",0.828117,0.171880,1
4,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",0.999997,4872508,Cost of Attendance. Cost of Attendance (COA) i...,0.170587,0.829411,1


In [4]:
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

if model_family == "cross_encoder":
    ce_model = CrossEncoder(model_name, max_length=max_length)
    model = ce_model.model.to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.mask_token_id is None:
        raise ValueError("Cross-encoder LOO requires a tokenizer with a mask token.")
    if tokenizer.pad_token_id is None:
        raise ValueError("Tokenizer must define a pad token.")

elif model_family in {"monot5", "duot5"}:
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    except Exception as e:
        raise ImportError(
            "Failed to load the T5 tokenizer. Install `sentencepiece` and `protobuf`, restart the kernel, and rerun."
        ) from e

    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    model.eval()

    decoder_start_token_id = model.config.decoder_start_token_id
    if decoder_start_token_id is None:
        decoder_start_token_id = tokenizer.pad_token_id

    true_ids = tokenizer.encode("true", add_special_tokens=False)
    false_ids = tokenizer.encode("false", add_special_tokens=False)
    if len(true_ids) != 1 or len(false_ids) != 1:
        raise ValueError("Expected 'true' and 'false' to map to single tokens.")
    true_token_id = int(true_ids[0])
    false_token_id = int(false_ids[0])
else:
    raise ValueError(f"Unsupported model family: {model_family}")

print(f"Loaded: {model_name}")
print(f"Device: {device}")


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Loaded: castorini/duot5-base-msmarco
Device: mps


In [5]:
def merge_wordpiece_spans(tokens, positions):
    spans = []
    current_tokens, current_positions = [], []
    for token, pos in zip(tokens, positions):
        if token.startswith("##") and current_tokens:
            current_tokens.append(token)
            current_positions.append(int(pos))
        else:
            if current_tokens:
                spans.append((current_tokens, current_positions))
            current_tokens = [token]
            current_positions = [int(pos)]
    if current_tokens:
        spans.append((current_tokens, current_positions))
    words = []
    for toks, poss in spans:
        word = toks[0]
        for tok in toks[1:]:
            word += tok.replace("##", "")
        words.append({"word": word, "positions": poss, "tokens": toks})
    return words


def merge_sentencepiece_spans(tokens, positions):
    special_tokens = set(tokenizer.all_special_tokens)
    spans = []
    current_word, current_positions, current_tokens = "", [], []
    for token, pos in zip(tokens, positions):
        if token in special_tokens:
            if current_tokens:
                spans.append({"word": current_word or ''.join(current_tokens), "positions": current_positions, "tokens": current_tokens})
                current_word, current_positions, current_tokens = "", [], []
            continue
        if token.startswith("▁"):
            if current_tokens:
                spans.append({"word": current_word or ''.join(current_tokens), "positions": current_positions, "tokens": current_tokens})
            current_word = token.lstrip("▁") or token
            current_positions = [int(pos)]
            current_tokens = [token]
        else:
            current_word += token
            current_positions.append(int(pos))
            current_tokens.append(token)
    if current_tokens:
        spans.append({"word": current_word or ''.join(current_tokens), "positions": current_positions, "tokens": current_tokens})
    return spans


In [6]:
if model_family == "cross_encoder":
    def tokenize_ce(query, passage):
        encoded = tokenizer(query, passage, max_length=max_length, truncation=True, padding=False, return_tensors="pt")
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        token_type_ids = encoded.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
        sep_positions = (input_ids[0] == tokenizer.sep_token_id).nonzero(as_tuple=True)[0].tolist()
        sep_idx = sep_positions[0] if sep_positions else len(tokens) - 1
        last_sep_idx = sep_positions[-1] if sep_positions else len(tokens) - 1
        doc_positions = list(range(sep_idx + 1, last_sep_idx))
        doc_tokens = [tokens[p] for p in doc_positions]
        doc_words = merge_wordpiece_spans(doc_tokens, doc_positions)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "tokens": tokens,
            "doc_words": doc_words,
        }

    def score_ce(tok):
        kwargs = {"input_ids": tok["input_ids"], "attention_mask": tok["attention_mask"]}
        if tok["token_type_ids"] is not None:
            kwargs["token_type_ids"] = tok["token_type_ids"]
        with torch.no_grad():
            outputs = model(**kwargs)
        return float(outputs.logits.squeeze(-1).detach().cpu().item())

    def mask_positions(tok, positions):
        perturbed = {
            "input_ids": tok["input_ids"].clone(),
            "attention_mask": tok["attention_mask"].clone(),
            "token_type_ids": None if tok["token_type_ids"] is None else tok["token_type_ids"].clone(),
        }
        perturbed["input_ids"][0, positions] = tokenizer.mask_token_id
        return perturbed

elif model_family == "monot5":
    def mono_input(query, passage):
        return f"Query: {query} Document: {passage} Relevant:"

    def _tok_len(text):
        return len(tokenizer(text, add_special_tokens=False)["input_ids"])

    def tokenize_mono(query, passage):
        text = mono_input(query, passage)
        encoded = tokenizer(text, max_length=max_length, truncation=True, padding=False, return_tensors="pt")
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
        l1 = _tok_len("Query: ")
        l2 = _tok_len("Query: " + query)
        l3 = _tok_len("Query: " + query + " Document: ")
        l4 = _tok_len("Query: " + query + " Document: " + passage)
        doc_positions = [pos for pos in range(l3, min(l4, input_ids.shape[1]))]
        doc_tokens = [tokens[p] for p in doc_positions]
        doc_words = merge_sentencepiece_spans(doc_tokens, doc_positions)
        return {"input_ids": input_ids, "attention_mask": attention_mask, "tokens": tokens, "doc_words": doc_words}

    def score_t5(tok):
        decoder_input_ids = torch.full((1,1), decoder_start_token_id, dtype=torch.long, device=device)
        with torch.no_grad():
            outputs = model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"], decoder_input_ids=decoder_input_ids)
        logits = outputs.logits[:,0,:]
        tf_logits = logits[:, [false_token_id, true_token_id]]
        return float(torch.softmax(tf_logits, dim=-1)[:,1].detach().cpu().item())

    def mask_positions(tok, positions):
        perturbed = {"input_ids": tok["input_ids"].clone(), "attention_mask": tok["attention_mask"].clone()}
        perturbed["input_ids"][0, positions] = tokenizer.pad_token_id
        perturbed["attention_mask"][0, positions] = 0
        return perturbed

elif model_family == "duot5":
    def duo_input(query, doc0, doc1):
        return f"Query: {query} Document0: {doc0} Document1: {doc1} Relevant:"

    def _tok_len(text):
        return len(tokenizer(text, add_special_tokens=False)["input_ids"])

    def tokenize_duo(query, doc0, doc1):
        text = duo_input(query, doc0, doc1)
        encoded = tokenizer(text, max_length=max_length, truncation=True, padding=False, return_tensors="pt")
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
        c1 = "Query: "
        c2 = c1 + query
        c3 = c2 + " Document0: "
        c4 = c3 + doc0
        c5 = c4 + " Document1: "
        c6 = c5 + doc1
        doc0_positions = [pos for pos in range(_tok_len(c3), min(_tok_len(c4), input_ids.shape[1]))]
        doc1_positions = [pos for pos in range(_tok_len(c5), min(_tok_len(c6), input_ids.shape[1]))]
        doc0_words = merge_sentencepiece_spans([tokens[p] for p in doc0_positions], doc0_positions)
        doc1_words = merge_sentencepiece_spans([tokens[p] for p in doc1_positions], doc1_positions)
        return {"input_ids": input_ids, "attention_mask": attention_mask, "tokens": tokens, "doc0_words": doc0_words, "doc1_words": doc1_words}

    def score_t5(tok):
        decoder_input_ids = torch.full((1,1), decoder_start_token_id, dtype=torch.long, device=device)
        with torch.no_grad():
            outputs = model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"], decoder_input_ids=decoder_input_ids)
        logits = outputs.logits[:,0,:]
        return float((logits[:, true_token_id] - logits[:, false_token_id]).detach().cpu().item())

    def mask_positions(tok, positions):
        perturbed = {"input_ids": tok["input_ids"].clone(), "attention_mask": tok["attention_mask"].clone()}
        perturbed["input_ids"][0, positions] = tokenizer.pad_token_id
        perturbed["attention_mask"][0, positions] = 0
        return perturbed


In [7]:
def compute_loo_cross_encoder(query, passage_i, passage_j):
    tok_i = tokenize_ce(query, passage_i)
    tok_j = tokenize_ce(query, passage_j)
    score_i = score_ce(tok_i)
    score_j = score_ce(tok_j)
    original_g = score_i - score_j

    doc_i_rows = []
    for span in tok_i["doc_words"]:
        masked_i = mask_positions(tok_i, span["positions"])
        g_masked = score_ce(masked_i) - score_j
        doc_i_rows.append({"word": span["word"], "positions": span["positions"], "support_score": float(original_g - g_masked)})

    doc_j_rows = []
    for span in tok_j["doc_words"]:
        masked_j = mask_positions(tok_j, span["positions"])
        g_masked = score_i - score_ce(masked_j)
        doc_j_rows.append({"word": span["word"], "positions": span["positions"], "support_score": float(original_g - g_masked)})

    return {"method": "leave_one_out", "score": float(original_g), "doc_i": doc_i_rows, "doc_j": doc_j_rows}


def compute_loo_monot5(query, passage_i, passage_j):
    tok_i = tokenize_mono(query, passage_i)
    tok_j = tokenize_mono(query, passage_j)
    score_i = score_t5(tok_i)
    score_j = score_t5(tok_j)
    original_g = score_i - score_j

    doc_i_rows = []
    for span in tok_i["doc_words"]:
        masked_i = mask_positions(tok_i, span["positions"])
        g_masked = score_t5(masked_i) - score_j
        doc_i_rows.append({"word": span["word"], "positions": span["positions"], "support_score": float(original_g - g_masked)})

    doc_j_rows = []
    for span in tok_j["doc_words"]:
        masked_j = mask_positions(tok_j, span["positions"])
        g_masked = score_i - score_t5(masked_j)
        doc_j_rows.append({"word": span["word"], "positions": span["positions"], "support_score": float(original_g - g_masked)})

    return {"method": "leave_one_out", "score": float(original_g), "doc_i": doc_i_rows, "doc_j": doc_j_rows}


def compute_loo_duot5(query, passage_i, passage_j):
    tok = tokenize_duo(query, passage_i, passage_j)
    original_g = score_t5(tok)

    doc_i_rows = []
    for span in tok["doc0_words"]:
        masked = mask_positions(tok, span["positions"])
        g_masked = score_t5(masked)
        doc_i_rows.append({"word": span["word"], "positions": span["positions"], "support_score": float(original_g - g_masked)})

    doc_j_rows = []
    for span in tok["doc1_words"]:
        masked = mask_positions(tok, span["positions"])
        g_masked = score_t5(masked)
        doc_j_rows.append({"word": span["word"], "positions": span["positions"], "support_score": float(original_g - g_masked)})

    return {"method": "leave_one_out", "score": float(original_g), "doc_i": doc_i_rows, "doc_j": doc_j_rows}


In [8]:
test_row = pairs_df[pairs_df["correct_pref"] == 1].iloc[0]
if model_family == "cross_encoder":
    test_loo = compute_loo_cross_encoder(test_row["query"], test_row["passage_i"], test_row["passage_j"])
elif model_family == "monot5":
    test_loo = compute_loo_monot5(test_row["query"], test_row["passage_i"], test_row["passage_j"])
else:
    test_loo = compute_loo_duot5(test_row["query"], test_row["passage_i"], test_row["passage_j"])

print(f"Original {cfg['score_label']}: {test_loo['score']:.6f}")
print("Top doc_i supportive words:")
for row in sorted(test_loo["doc_i"], key=lambda x: x["support_score"], reverse=True)[:10]:
    print(f"  {row['word']:<20} {row['support_score']:.4f}")
print("\nTop doc_j supportive words:")
for row in sorted(test_loo["doc_j"], key=lambda x: x["support_score"], reverse=True)[:10]:
    print(f"  {row['word']:<20} {row['support_score']:.4f}")


Original pairwise margin (true - false): 11.248667
Top doc_i supportive words:
  Eastern              2.3929
  Illinois             0.8400
  year.                0.4085
  University           0.2145
  has                  0.1819
  is                   0.1018
  fees                 0.0946
  university           0.0831
  non-residents.       0.0780
  Admission            0.0777

Top doc_j supportive words:
  to                   0.2053
  Cost                 0.1902
  total                0.1570
  expenses             0.1551
  expenses.            0.1476
  fees                 0.1305
  cost                 0.1211
  to                   0.0994
  estimate             0.0912
  student              0.0851


In [9]:
records = []
failed = []

for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc=f"Computing LOO ({cfg['label']})"):
    try:
        if model_family == "cross_encoder":
            loo = compute_loo_cross_encoder(row["query"], row["passage_i"], row["passage_j"])
        elif model_family == "monot5":
            loo = compute_loo_monot5(row["query"], row["passage_i"], row["passage_j"])
        else:
            loo = compute_loo_duot5(row["query"], row["passage_i"], row["passage_j"])
        records.append({
            "qid": row["qid"],
            "query": row["query"],
            "pid_i": row["pid_i"],
            "pid_j": row["pid_j"],
            "g_score": row["g_score"],
            "correct_pref": row["correct_pref"],
            "leave_one_out": loo,
        })
    except Exception as e:
        failed.append({"qid": row["qid"], "pid_i": row["pid_i"], "pid_j": row["pid_j"], "error": str(e)})
        print(f"Error on qid={row['qid']} pid_i={row['pid_i']} pid_j={row['pid_j']}: {e}")

print(f"Computed LOO explanations for {len(records)} pairs")
if failed:
    print(f"Failed pairs: {len(failed)}")


Computing LOO (DuoT5):   0%|          | 0/80 [00:00<?, ?it/s]

Computed LOO explanations for 80 pairs


In [10]:
with open(loo_out, "wb") as f:
    pickle.dump(records, f)
print(f"Saved LOO records -> {loo_out}")

summary_rows = []
for record in records:
    doc_i = record["leave_one_out"]["doc_i"]
    doc_j = record["leave_one_out"]["doc_j"]
    best_i = max(doc_i, key=lambda x: x["support_score"]) if doc_i else {"word": "", "support_score": np.nan}
    best_j = max(doc_j, key=lambda x: x["support_score"]) if doc_j else {"word": "", "support_score": np.nan}
    summary_rows.append({
        "qid": record["qid"],
        "pid_i": record["pid_i"],
        "pid_j": record["pid_j"],
        "g_score": record["g_score"],
        "loo_score": record["leave_one_out"]["score"],
        "doc_i_top_word": best_i["word"],
        "doc_i_top_score": best_i["support_score"],
        "doc_j_top_word": best_j["word"],
        "doc_j_top_score": best_j["support_score"],
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(loo_summary_out, index=False)
print(f"Saved LOO summary -> {loo_summary_out}")
summary_df.head()


Saved LOO records -> ../outputs_t5/loo_attributions.pkl
Saved LOO summary -> ../outputs_t5/loo_attributions_summary.csv


,qid,pid_i,pid_j,g_score,loo_score,doc_i_top_word,doc_i_top_score,doc_j_top_word,doc_j_top_score
0,1049774,7185662,4083953,0.615020,11.248667,Eastern,2.392871,to,0.205262
1,1049774,7185662,2281863,0.610489,14.094520,Eastern,1.421126,Parkland,0.230493
2,1049774,7185662,6262988,0.767897,14.111420,Eastern,2.713932,Southern,1.191164
3,1049774,7185662,6650510,0.171880,12.306589,Eastern,4.975700,Urbana,1.378067
4,1049774,7185662,4872508,0.829411,13.398839,Eastern,1.358285,(COA),0.217832
